# Hydrograph Investigation

Quick notebook for a single example. It loads the observed upstream/downstream gauge hydrographs, lists the available independent and compound RAPID states for that example, and plots:

1. Observed upstream vs downstream gauge hydrographs
2. Observed gauges plus selected RAPID outlet hydrographs

By default the RAPID plot includes all available independent and compound states for the example. Set `SELECTED_STATE_IDS` to a smaller list if needed.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import qualitative


def find_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "network_variants").exists() and (candidate / "SWORD_gauge_match").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from the current working directory.")


REPO_ROOT = find_repo_root()

# ----------------------------
# notebook configuration
# ----------------------------
EXAMPLE_ID = 3
VARIANT_KIND = "anchored"  # "anchored" or "realized"
INCLUDE_BASE = True
INCLUDE_INDEP = True
INCLUDE_COMPOUND = True
SELECTED_STATE_IDS = None  # None => include all available states for this example
MAX_RAPID_STATES = None    # e.g. 10 to limit the overlay plot

# Optional manual peak overrides. Leave as None to use the max discharge within the selected period.
UPSTREAM_PEAK_TIME = None
DOWNSTREAM_PEAK_TIME = None

MASK_XLSX = REPO_ROOT / "SWORD_gauge_match/outputs/mask_hydro_quality.xlsx"
TIMESERIES_PATHS = [
    REPO_ROOT / "SWORD_gauge_match/outputs/subdaily_timeseries.csv",
    REPO_ROOT / "SWORD_gauge_match/outputs/subdaily_timeseries.parquet",
    REPO_ROOT / "SWORD_gauge_match/outputs/gauge_timeseries.parquet",
    REPO_ROOT / "SWORD_gauge_match/outputs/grdc_timeseries.parquet",
]

example_tag = f"{EXAMPLE_ID:02d}"
print("Repo root:", REPO_ROOT)
print("Example:", example_tag)
print("Variant kind:", VARIANT_KIND)


Repo root: /Users/6256481/Code/river-hierarchy
Example: 03
Variant kind: anchored


In [2]:
def load_quality_row(example_id: int, mask_xlsx: Path) -> pd.Series:
    quality = pd.read_excel(mask_xlsx, sheet_name="Hydrograph_quality")
    row = quality.loc[quality["Example"] == example_id]
    if row.empty:
        raise ValueError(f"No hydrograph-quality row found for Example={example_id}")
    return row.iloc[0]


def load_station_timeseries(station_key: str, paths: list[Path]) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in paths:
        if not path.exists():
            continue

        try:
            if path.suffix == ".csv":
                df = pd.read_csv(path)
            elif path.suffix == ".parquet":
                try:
                    df = pd.read_parquet(path, filters=[("station_key", "==", station_key)])
                except Exception:
                    df = pd.read_parquet(path)
            else:
                continue
        except Exception:
            continue

        if "station_key" not in df.columns:
            continue

        df = df.loc[df["station_key"].astype(str) == str(station_key)].copy()
        if df.empty:
            continue

        if "discharge_cms" in df.columns and "discharge" not in df.columns:
            df = df.rename(columns={"discharge_cms": "discharge"})

        if "time" not in df.columns or "discharge" not in df.columns:
            continue

        df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
        df["discharge"] = pd.to_numeric(df["discharge"], errors="coerce")
        df = df.dropna(subset=["time", "discharge"])
        frames.append(df[["station_key", "time", "discharge"]])

    if not frames:
        raise ValueError(f"No timeseries found for station {station_key}")

    return (
        pd.concat(frames, ignore_index=True)
        .sort_values("time")
        .drop_duplicates(subset=["station_key", "time"], keep="first")
        .reset_index(drop=True)
    )


def discover_experiment_dirs(example_tag: str, variant_kind: str) -> dict[str, Path]:
    root = REPO_ROOT / "network_variants" / "output"
    dirs = {
        "indep": root / f"{example_tag}_indep__{variant_kind}",
        "compound": root / f"{example_tag}_compound_bubbles__{variant_kind}",
    }
    return {key: value for key, value in dirs.items() if value.exists()}


def build_state_catalog(experiment_dirs: dict[str, Path]) -> pd.DataFrame:
    rows: list[dict] = []
    base_added = False

    for mode, exp_dir in experiment_dirs.items():
        states_dir = exp_dir / "states"
        if not states_dir.exists():
            continue

        for state_dir in sorted(p for p in states_dir.iterdir() if p.is_dir()):
            state_id = state_dir.name
            manifest_path = state_dir / "variant" / "summary" / "variant_manifest.json"
            hydro_path = state_dir / "rapid" / "run" / "outlet_hydrograph.csv"
            if not manifest_path.exists() or not hydro_path.exists():
                continue

            if state_id == "S000_base" and base_added:
                continue

            manifest = json.loads(manifest_path.read_text())
            selected_units = manifest.get("selected_unit_ids", []) or []
            selected_units = [int(x) for x in selected_units]

            if state_id == "S000_base":
                state_kind = "base"
                display_label = "S000_base"
                base_added = True
            elif mode == "compound" or state_id.startswith("CB"):
                state_kind = "compound"
                display_label = f"{state_id} | compound units={selected_units}"
            else:
                state_kind = "unit"
                display_label = f"{state_id} | units={selected_units}"

            rows.append(
                {
                    "mode": mode,
                    "state_kind": state_kind,
                    "state_id": state_id,
                    "selected_unit_ids": selected_units,
                    "selected_unit_ids_text": ", ".join(str(x) for x in selected_units),
                    "n_selected_units": len(selected_units),
                    "display_label": display_label,
                    "state_dir": state_dir,
                    "hydro_path": hydro_path,
                }
            )

    if not rows:
        raise ValueError("No RAPID hydrograph outputs were found for the selected example.")

    catalog = pd.DataFrame(rows)
    kind_order = {"base": 0, "unit": 1, "compound": 2}
    catalog["kind_order"] = catalog["state_kind"].map(kind_order)
    return catalog.sort_values(["kind_order", "mode", "state_id"]).reset_index(drop=True)


def select_states(catalog: pd.DataFrame) -> pd.DataFrame:
    selected = catalog.copy()

    if not INCLUDE_BASE:
        selected = selected.loc[selected["state_kind"] != "base"]
    if not INCLUDE_INDEP:
        selected = selected.loc[selected["mode"] != "indep"]
    if not INCLUDE_COMPOUND:
        selected = selected.loc[selected["mode"] != "compound"]
    if SELECTED_STATE_IDS is not None:
        selected = selected.loc[selected["state_id"].isin(SELECTED_STATE_IDS)]
    if MAX_RAPID_STATES is not None:
        selected = selected.head(MAX_RAPID_STATES)

    return selected.reset_index(drop=True)


def load_outlet_hydrograph(hydro_path: Path) -> pd.DataFrame:
    hydro = pd.read_csv(hydro_path)
    hydro["time_utc"] = pd.to_datetime(hydro["time_utc"], utc=True, errors="coerce")
    hydro["q_outlet_cms"] = pd.to_numeric(hydro["q_outlet_cms"], errors="coerce")
    hydro = hydro.dropna(subset=["time_utc", "q_outlet_cms"]).copy()
    return hydro.sort_values("time_utc").reset_index(drop=True)


def choose_peak_time(frame: pd.DataFrame, manual_time=None) -> pd.Timestamp:
    if manual_time is None:
        idx = frame["discharge"].idxmax()
        return pd.Timestamp(frame.loc[idx, "time"])

    target = pd.Timestamp(manual_time)
    if target.tzinfo is None:
        target = target.tz_localize("UTC")
    else:
        target = target.tz_convert("UTC")

    nearest_idx = (frame["time"] - target).abs().idxmin()
    return pd.Timestamp(frame.loc[nearest_idx, "time"])


def add_peak_lines(fig: go.Figure, upstream_peak: pd.Timestamp, downstream_peak: pd.Timestamp) -> None:
    fig.add_vline(x=upstream_peak, line_dash="dash", line_color="#1f77b4", opacity=0.9)
    fig.add_vline(x=downstream_peak, line_dash="dash", line_color="#d62728", opacity=0.9)
    fig.add_annotation(x=upstream_peak, y=1.0, yref="paper", text="Up peak", showarrow=False, yshift=10, font=dict(color="#1f77b4"))
    fig.add_annotation(x=downstream_peak, y=1.0, yref="paper", text="Down peak", showarrow=False, yshift=10, font=dict(color="#d62728"))


def print_example_lists(catalog: pd.DataFrame) -> None:
    indep = catalog.loc[catalog["mode"] == "indep", ["state_id", "selected_unit_ids_text"]]
    compound = catalog.loc[catalog["mode"] == "compound", ["state_id", "selected_unit_ids_text"]]

    print("Independent states:")
    if indep.empty:
        print("  none")
    else:
        for _, row in indep.iterrows():
            print(f"  {row['state_id']}: [{row['selected_unit_ids_text']}]")

    print("\nCompound states:")
    if compound.empty:
        print("  none")
    else:
        for _, row in compound.iterrows():
            print(f"  {row['state_id']}: [{row['selected_unit_ids_text']}]")


In [18]:
catalog.iloc[[0, 23, 14, 30]]

,mode,state_kind,state_id,selected_unit_ids,selected_unit_ids_text,n_selected_units,display_label,state_dir,hydro_path,kind_order
0,indep,base,S000_base,[],,0,S000_base,/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,0
23,indep,unit,S024_unit_15,[15],15,1,S024_unit_15 | units=[15],/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,1
14,indep,unit,S015_unit_17,[17],17,1,S015_unit_17 | units=[17],/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,1
30,indep,unit,S031_unit_16,[16],16,1,S031_unit_16 | units=[16],/Users/6256481/Code/river-hierarchy/network_va...,/Users/6256481/Code/river-hierarchy/network_va...,1


In [19]:
quality_row = load_quality_row(EXAMPLE_ID, MASK_XLSX)
up_station = quality_row["Up"]
down_station = quality_row["Down"]
period_start = pd.to_datetime(quality_row["Period_start"], utc=True)
period_end = pd.to_datetime(quality_row["Period_end"], utc=True)

experiment_dirs = discover_experiment_dirs(example_tag, VARIANT_KIND)
catalog = build_state_catalog(experiment_dirs)

catalog1 = catalog.iloc[[0, 23, 14, 30]]
selected_catalog = select_states(catalog1)

up_ts = load_station_timeseries(up_station, TIMESERIES_PATHS)
down_ts = load_station_timeseries(down_station, TIMESERIES_PATHS)
up_ts = up_ts.loc[up_ts["time"].between(period_start, period_end)].reset_index(drop=True)
down_ts = down_ts.loc[down_ts["time"].between(period_start, period_end)].reset_index(drop=True)

up_peak_time = choose_peak_time(up_ts, UPSTREAM_PEAK_TIME)
down_peak_time = choose_peak_time(down_ts, DOWNSTREAM_PEAK_TIME)

print("Up station:", up_station)
print("Down station:", down_station)
print("Period:", period_start, "to", period_end)
print("Detected experiment dirs:")
for mode, exp_dir in experiment_dirs.items():
    print(f"  {mode}: {exp_dir}")
print("\nSelected upstream peak:", up_peak_time)
print("Selected downstream peak:", down_peak_time)
print("\nAvailable state catalog:")
display(catalog[["mode", "state_kind", "state_id", "selected_unit_ids_text", "display_label"]])
print_example_lists(catalog)

if selected_catalog.empty:
    raise ValueError("The current selection produced no RAPID states to plot.")

rapid_series = []
for row in selected_catalog.itertuples(index=False):
    hydro = load_outlet_hydrograph(Path(row.hydro_path))
    rapid_series.append(
        {
            "mode": row.mode,
            "state_kind": row.state_kind,
            "state_id": row.state_id,
            "display_label": row.display_label,
            "hydro": hydro,
        }
    )

selected_catalog[["mode", "state_kind", "state_id", "selected_unit_ids_text", "display_label"]]


/var/folders/bs/xx1t3fn91jl3k26fn1pndd_c0000gn/T/ipykernel_25371/962997219.py:17: DtypeWarning: Columns (1,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
/var/folders/bs/xx1t3fn91jl3k26fn1pndd_c0000gn/T/ipykernel_25371/962997219.py:17: DtypeWarning: Columns (1,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Up station: BR:3652880
Down station: BR:3652890
Period: 2023-02-13 00:01:00+00:00 to 2023-02-22 00:00:00+00:00
Detected experiment dirs:
  indep: /Users/6256481/Code/river-hierarchy/network_variants/output/03_indep__anchored
  compound: /Users/6256481/Code/river-hierarchy/network_variants/output/03_compound_bubbles__anchored

Selected upstream peak: 2023-02-17 08:15:00+00:00
Selected downstream peak: 2023-02-17 17:45:00+00:00

Available state catalog:


,mode,state_kind,state_id,selected_unit_ids_text,display_label
0,indep,base,S000_base,,S000_base
1,indep,unit,S001_unit_5,5,S001_unit_5 | units=[5]
2,indep,unit,S002_unit_1,1,S002_unit_1 | units=[1]
3,indep,unit,S003_unit_4,4,S003_unit_4 | units=[4]
4,indep,unit,S004_unit_21,21,S004_unit_21 | units=[21]
5,indep,unit,S005_unit_18,18,S005_unit_18 | units=[18]
6,indep,unit,S006_unit_20,20,S006_unit_20 | units=[20]
7,indep,unit,S007_unit_14,14,S007_unit_14 | units=[14]
8,indep,unit,S008_unit_23,23,S008_unit_23 | units=[23]
9,indep,unit,S009_unit_31,31,S009_unit_31 | units=[31]


Independent states:
  S000_base: []
  S001_unit_5: [5]
  S002_unit_1: [1]
  S003_unit_4: [4]
  S004_unit_21: [21]
  S005_unit_18: [18]
  S006_unit_20: [20]
  S007_unit_14: [14]
  S008_unit_23: [23]
  S009_unit_31: [31]
  S010_unit_13: [13]
  S011_unit_3: [3]
  S012_unit_25: [25]
  S014_unit_19: [19]
  S015_unit_17: [17]
  S016_unit_22: [22]
  S017_unit_7: [7]
  S018_unit_9: [9]
  S019_unit_29: [29]
  S020_unit_28: [28]
  S021_unit_27: [27]
  S022_unit_12: [12]
  S023_unit_24: [24]
  S024_unit_15: [15]
  S025_unit_11: [11]
  S026_unit_6: [6]
  S027_unit_8: [8]
  S028_unit_26: [26]
  S029_unit_10: [10]
  S030_unit_30: [30]
  S031_unit_16: [16]

Compound states:
  CB001_bubble_14: [22, 23, 24, 25, 26, 27, 28, 29]
  CB002_bubble_9: [8, 9, 10, 14]
  CB003_bubble_10: [15, 16, 17]
  CB004_bubble_1: [11, 12]
  CB005_bubble_4: [2, 6]
  CB006_bubble_12: [19, 20]
  CB007_bubble_15: [30, 31]


,mode,state_kind,state_id,selected_unit_ids_text,display_label
0,indep,base,S000_base,,S000_base
1,indep,unit,S024_unit_15,15,S024_unit_15 | units=[15]
2,indep,unit,S015_unit_17,17,S015_unit_17 | units=[17]
3,indep,unit,S031_unit_16,16,S031_unit_16 | units=[16]


In [20]:
fig_obs = go.Figure()
fig_obs.add_trace(
    go.Scatter(
        x=up_ts["time"],
        y=up_ts["discharge"],
        mode="lines",
        name=f"Observed upstream: {up_station}",
        line=dict(color="#1f77b4", width=3),
    )
)
fig_obs.add_trace(
    go.Scatter(
        x=down_ts["time"],
        y=down_ts["discharge"],
        mode="lines",
        name=f"Observed downstream: {down_station}",
        line=dict(color="#d62728", width=3),
    )
)
add_peak_lines(fig_obs, up_peak_time, down_peak_time)
fig_obs.update_layout(
    title=f"Example {example_tag}: observed upstream vs downstream gauges",
    xaxis_title="Time (UTC)",
    yaxis_title="Discharge (cms)",
    template="plotly_white",
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_obs.show()

fig_rapid = go.Figure()
fig_rapid.add_trace(
    go.Scatter(
        x=up_ts["time"],
        y=up_ts["discharge"],
        mode="lines",
        name=f"Observed upstream: {up_station}",
        line=dict(color="#1f77b4", width=3),
    )
)
fig_rapid.add_trace(
    go.Scatter(
        x=down_ts["time"],
        y=down_ts["discharge"],
        mode="lines",
        name=f"Observed downstream: {down_station}",
        line=dict(color="#d62728", width=3),
    )
)

rapid_palette = qualitative.Dark24 + qualitative.Alphabet + qualitative.Set3
for i, item in enumerate(rapid_series):
    hydro = item["hydro"]
    if item["state_kind"] == "base":
        color = "#111111"
        width = 3
        opacity = 1.0
    else:
        color = rapid_palette[i % len(rapid_palette)]
        width = 1.8
        opacity = 0.75

    fig_rapid.add_trace(
        go.Scatter(
            x=hydro["time_utc"],
            y=hydro["q_outlet_cms"],
            mode="lines",
            name=item["display_label"],
            line=dict(color=color, width=width),
            opacity=opacity,
        )
    )

add_peak_lines(fig_rapid, up_peak_time, down_peak_time)
fig_rapid.update_layout(
    title=f"Example {example_tag}: observed gauges + RAPID outlet hydrographs",
    xaxis_title="Time (UTC)",
    yaxis_title="Discharge (cms)",
    template="plotly_white",
    height=700,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
)
fig_rapid.show()
